# 08 — New Distance Approach: Related Metric Scaling (RelMS)\n",
    "\n",
    "Uses `related_metric_scaling_dist_matrix_faster` from `robust_mixed_dist.mixed`\n",
    "to compute a single combined distance matrix from all 5 image-type PCA blocks.\n",
    "\n",
    "RelMS normalises the distance matrix by its **geometric variability** (a measure of\n",
    "spread) before combining — making distances comparable across different scales.\n",
    "This differs from simple Euclidean distance on the concatenated 550 features.\n",
    "\n",
    "| Step | What happens |\n",
    "|------|--------------|\n",
    "| 1 | Install / verify `robust-mixed-dist` |\n",
    "| 2 | Load 5 PCA CSVs → concatenate → X (416 × 550) |\n",
    "| 3 | Compute D (416 × 416) via `related_metric_scaling_dist_matrix_faster` |\n",
    "| 4 | Cluster with `kmedoids.fasterpam` for k=3 and k=4 |\n",
    "| 5 | Evaluate: Spread · Silhouette · ARI · NMI · CDR crosstab |\n",
    "| 6 | Save `results_new_distance_5images.csv` |

## Step 1 — Install and verify package version

In [4]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-600:] if r.stderr else "(no stderr)")

run_pip("robust-mixed-dist==0.1.26")
for pkg in ["kmedoids", "openpyxl"]:
    run_pip(pkg)
print("Done.")

import importlib.metadata
print(f"\nrobust-mixed-dist version: {importlib.metadata.version('robust-mixed-dist')}")

  OK    robust-mixed-dist==0.1.26
  OK    kmedoids
  OK    openpyxl
Done.

robust-mixed-dist version: 0.1.26


## Imports

In [5]:
import os
import numpy as np
import pandas as pd

import kmedoids
# function lives in robust_mixed_dist.mixed, not at the package top level
from robust_mixed_dist.mixed import related_metric_scaling_dist_matrix_faster
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Step 2 — Load the 5 PCA files and build X

Each file is 416 × 111 (patient_id + 110 PCA components).  
Drop `patient_id` and concatenate horizontally in the order:
cor → gfc_sag → gfc_tra → masked_tra → sbj_sag.  
Result: X with shape **(416, 550)**.

In [6]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs (pca_*.csv, clinical xlsx)
NB_PADDED = OUTPUT_DIR          # legacy alias: this notebook's own outputs

PCA_FILES = [
    "pca_cor.csv",
    "pca_gfc_sag.csv",
    "pca_gfc_tra.csv",
    "pca_masked_tra.csv",
    "pca_sbj_sag.csv",
]

blocks      = []   # list of (416, 110) arrays, one per image type
patient_ids = None

for fname in PCA_FILES:
    df = pd.read_csv(os.path.join(NB_DIR, fname))
    if patient_ids is None:
        patient_ids = df["patient_id"].values
    pca_cols = [c for c in df.columns if c != "patient_id"]
    blocks.append(df[pca_cols].to_numpy(dtype=np.float64))
    print(f"  {fname:<25} → block shape {blocks[-1].shape}")

X = np.concatenate(blocks, axis=1)
print(f"\nCombined X shape  : {X.shape}  (patients × PCA features)")
print(f"Block sizes passed to RelMS: {[b.shape[1] for b in blocks]}")

  pca_cor.csv               → block shape (416, 110)
  pca_gfc_sag.csv           → block shape (416, 110)
  pca_gfc_tra.csv           → block shape (416, 110)
  pca_masked_tra.csv        → block shape (416, 110)
  pca_sbj_sag.csv           → block shape (416, 110)

Combined X shape  : (416, 550)  (patients × PCA features)
Block sizes passed to RelMS: [110, 110, 110, 110, 110]


## Load CDR for evaluation

CDR is **not** used during clustering — only for post-hoc evaluation.

In [7]:
df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})

df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)

cdr_series = df_clin.set_index("patient_id")["CDR"].reindex(patient_ids)
mask_cdr   = cdr_series.notnull().values
cdr_int    = (cdr_series.values[mask_cdr] * 2).astype(int)

print(f"Clinical table    : {df_clin.shape}")
print(f"Patients with CDR : {int(mask_cdr.sum())} / {len(patient_ids)}")
print(f"CDR categories    : {sorted(set(cdr_int))}  (CDR × 2)")

Clinical table    : (416, 12)
Patients with CDR : 235 / 416
CDR categories    : [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]  (CDR × 2)


## Step 3 — Compute combined distance matrix using RelMS\n",
    "\n",
    "In version 0.1.26 the professor updated `related_metric_scaling_dist_matrix_faster`\n",
    "so that `p1` can be **a list of block sizes** — one integer per image type.\n",
    "Each block gets its own sub-distance matrix, normalised by its geometric variability,\n",
    "then all 5 are combined into one final (416 × 416) matrix.\n",
    "\n",
    "| Parameter | Value | Meaning |\n",
    "|-----------|-------|---------|\n",
    "| `p1` | `[110, 110, 110, 110, 110]` | 5 quantitative blocks, each 110 PCA components (cor, gfc_sag, gfc_tra, masked_tra, sbj_sag) |\n",
    "| `p2` | `0` | No binary columns |\n",
    "| `p3` | `0` | No multiclass columns |\n",
    "| `d1` | `\"euclidean\"` | Euclidean distance within each block |\n",
    "| `d2` | `\"sokal\"` | Not used when p2=0 but required by the API |\n",
    "| `d3` | `\"hamming\"` | Not used when p3=0 but required by the API |\n",
    "| `weights` | `None` | All 5 blocks weighted equally |

In [9]:
print("Computing RelMS distance matrix (5 blocks × 110 components each) ...")

# The function expects a pandas DataFrame — numpy arrays trigger a bug in 0.1.26
# where X_arr is only assigned for DataFrame inputs.  Wrapping fixes it.
X_df = pd.DataFrame(X)

D = related_metric_scaling_dist_matrix_faster(
    X=X_df,
    p1=[110, 110, 110, 110, 110],  # 5 blocks: cor, gfc_sag, gfc_tra, masked_tra, sbj_sag
    p2=0,
    p3=0,
    d1="euclidean",
    d2="sokal",
    d3="hamming",
    weights=None,
)

print(f"Distance matrix D : {D.shape}")
print(f"Value range       : [{D.min():.4f}, {D.max():.4f}]")
print(f"Is symmetric      : {np.allclose(D, D.T)}")
print(f"Diagonal is zero  : {np.allclose(np.diag(D), 0)}")

Computing RelMS distance matrix (5 blocks × 110 components each) ...
Distance matrix D : (416, 416)
Value range       : [0.0000, 3.9054]
Is symmetric      : True
Diagonal is zero  : True


## Step 4 — Cluster with FasterPAM (k=3 and k=4)

`kmedoids.fasterpam` is used directly because `SampleDistClustering` does not
accept a precomputed distance matrix — it always recomputes D from raw X.
Here we already have D, so we bypass the wrapper entirely.

In [10]:
all_results = []

for k in [3, 4]:
    result = kmedoids.fasterpam(D, medoids=k, random_state=42)
    labels = np.array(result.labels, dtype=int)

    sizes = {c: int((labels == c).sum()) for c in sorted(set(labels))}
    print(f"\nk={k}  |  loss={result.loss:.4f}  |  sizes={sizes}")

    # ── % healthy per cluster (CDR = 0.0) ────────────────────────────────────
    df_eval  = pd.DataFrame({"cluster": labels, "CDR": cdr_series.values})
    df_eval  = df_eval[df_eval["CDR"].notnull()].copy()
    pct_tab  = pd.crosstab(
        df_eval["cluster"], df_eval["CDR"], normalize="index"
    ).mul(100).round(1)
    pct_healthy = pct_tab[0.0] if 0.0 in pct_tab.columns \
                  else pd.Series(0.0, index=pct_tab.index)
    spread = float(pct_healthy.max() - pct_healthy.min())

    # ── CDR crosstab ──────────────────────────────────────────────────────────
    df_ev2 = df_eval.copy()
    df_ev2["CDR"] = df_ev2["CDR"].astype(str)
    ct = pd.crosstab(df_ev2["cluster"], df_ev2["CDR"],
                     margins=True, margins_name="Total")
    print("  Crosstab (count):")
    print(ct.to_string())
    print("  Row % (CDR proportion within each cluster):")
    print(pct_tab.to_string())

    # ── Silhouette ────────────────────────────────────────────────────────────
    sil = silhouette_score(D, labels, metric="precomputed")

    # ── ARI / NMI ─────────────────────────────────────────────────────────────
    ari = adjusted_rand_score(cdr_int, labels[mask_cdr])
    nmi = normalized_mutual_info_score(cdr_int, labels[mask_cdr])

    print(f"\n  spread={spread:.2f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

    all_results.append({
        "method":             "relms",
        "k":                  k,
        "silhouette":         round(sil, 4),
        "ari":                round(ari, 4),
        "nmi":                round(nmi, 4),
        "spread_pct_healthy": round(spread, 2),
    })

print("\nClustering complete.")


k=3  |  loss=1144.4943  |  sizes={np.int64(0): 135, np.int64(1): 115, np.int64(2): 166}
  Crosstab (count):
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         50   23    8    2     83
1         36   12    4    0     52
2         49   35   16    0    100
Total    135   70   28    2    235
  Row % (CDR proportion within each cluster):
CDR       0.0   0.5   1.0  2.0
cluster                       
0        60.2  27.7   9.6  2.4
1        69.2  23.1   7.7  0.0
2        49.0  35.0  16.0  0.0

  spread=20.20%  sil=0.0183  ARI=-0.0103  NMI=0.0233

k=4  |  loss=1133.5998  |  sizes={np.int64(0): 105, np.int64(1): 107, np.int64(2): 130, np.int64(3): 74}
  Crosstab (count):
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         34   17    5    0     56
1         35   12    3    0     50
2         42   27   13    0     82
3         24   14    7    2     47
Total    135   70   28    2    235
  Row % (CDR proportion within each cluster):
CDR      

## Step 5 — Summary table and export

In [11]:
df_results = pd.DataFrame(all_results)

print("RelMS combined distance — 5 image types — FasterPAM")
print("=" * 60)
print(df_results.to_string(index=False))
print(f"\nSilhouette: all {len(patient_ids)} patients (precomputed D)")
print(f"ARI / NMI : {mask_cdr.sum()} patients with CDR  (CDR × 2 integer categories)")

out_path = os.path.join(NB_PADDED, "results_new_distance_5images.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

RelMS combined distance — 5 image types — FasterPAM
method  k  silhouette     ari    nmi  spread_pct_healthy
 relms  3      0.0183 -0.0103 0.0233                20.2
 relms  4      0.0106 -0.0056 0.0240                18.9

Silhouette: all 416 patients (precomputed D)
ARI / NMI : 235 patients with CDR  (CDR × 2 integer categories)

Saved: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\new_results_with_padding\results_new_distance_5images.csv
